# Top-fragment sequence export

Load every fragment from the top 10 clusters and add the adjacent amino-acid strips from its parent sequence.

In [3]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display


TOP_CLUSTER_COUNT = 10
FLANK_LENGTH = 10
SAVE_CSV = True


def repo_root_from_here() -> Path:
    here = Path.cwd().resolve()
    for path in (here, *here.parents):
        if (path / "sequence_analysis").is_dir():
            return path
    raise FileNotFoundError("Could not locate the repository root.")


def require_columns(frame: pd.DataFrame, columns: set[str], source: Path) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise ValueError(f"{source} is missing required columns: {missing}")


def latest_profile_csv(data_dir: Path) -> Path:
    filename = "window_sequence_profiles.csv"
    candidates = list((data_dir / "results" / "runs").glob(f"*/{filename}"))
    if candidates:
        return max(candidates, key=lambda path: path.stat().st_mtime_ns)
    legacy_path = data_dir / "results" / filename
    if legacy_path.is_file():
        return legacy_path
    raise FileNotFoundError(f"Could not find {filename}")


def protein_dngc(profile: object, delimiter: object) -> float:
    values = np.fromstring(str(profile), sep=str(delimiter or ";"))
    active = values[np.isfinite(values) & (values > 0)]
    return float(active.mean()) if active.size else 0.0


def load_parent_dngc(path: Path) -> pd.Series:
    columns = ["id", "hyperplane_distance_mean_profile", "profile_delimiter"]
    frame = pd.read_csv(path, usecols=columns)
    scores = [
        protein_dngc(profile, delimiter)
        for profile, delimiter in zip(
            frame["hyperplane_distance_mean_profile"],
            frame["profile_delimiter"].fillna(";"),
            strict=True,
        )
    ]
    return pd.Series(scores, index=frame["id"].astype(str)).groupby(level=0).first()


def parse_position(value: object) -> tuple[int, int]:
    match = re.fullmatch(r"\s*(\d+)\s*[-–—]\s*(\d+)\s*", str(value))
    if match is None:
        raise ValueError(f"Invalid fragment position: {value!r}")
    start, end = map(int, match.groups())
    if start < 1 or end < start:
        raise ValueError(f"Invalid fragment position: {value!r}")
    return start, end


def parent_id_for(fragment: pd.Series, start: int) -> str:
    if "protein_id" in fragment.index and pd.notna(fragment["protein_id"]):
        return str(fragment["protein_id"])
    fragment_id = str(fragment["id"])
    suffix = f"-{start}"
    if not fragment_id.endswith(suffix):
        raise ValueError(f"Cannot infer parent ID from fragment ID {fragment_id!r}")
    return fragment_id.removesuffix(suffix)


def sequence_record(
    fragment: pd.Series,
    parent_id: str,
    parent_dngc: float,
    sequence_type: str,
    sequence: str,
) -> dict[str, object]:
    return {
        "parent_name": fragment["name"],
        "parent_id": parent_id,
        "cluster": fragment["cluster"],
        "DNGC": parent_dngc,
        "fragment_DNGC": fragment["DNGC"],
        "id": fragment["id"],
        "position": fragment["position"],
        "sequence_type": sequence_type,
        "sequence": sequence,
        "length": len(sequence),
    }


def fragment_sequence_rows(
    fragments: pd.DataFrame,
    parents: pd.Series,
    parent_dngc: pd.Series,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for _, fragment in fragments.iterrows():
        start, end = parse_position(fragment["position"])
        parent_id = parent_id_for(fragment, start)
        if parent_id not in parents.index:
            raise KeyError(f"Parent sequence not found: {parent_id}")
        if parent_id not in parent_dngc.index:
            raise KeyError(f"Parent DNGC not found: {parent_id}")
        parent_sequence = parents.at[parent_id]
        dngc = float(parent_dngc.at[parent_id])
        existing_sequence = parent_sequence[start - 1 : end]
        expected_sequence = str(fragment["sequence"])
        if existing_sequence != expected_sequence:
            raise ValueError(f"Fragment sequence does not match parent sequence: {fragment['id']}")

        before_sequence = parent_sequence[max(0, start - 1 - FLANK_LENGTH) : start - 1]
        after_sequence = parent_sequence[end : end + FLANK_LENGTH]
        if before_sequence:
            rows.append(
                sequence_record(
                    fragment,
                    parent_id,
                    dngc,
                    "before_sequence",
                    before_sequence + existing_sequence,
                )
            )
        rows.append(
            sequence_record(fragment, parent_id, dngc, "existing_sequence", existing_sequence)
        )
        if after_sequence:
            rows.append(
                sequence_record(
                    fragment,
                    parent_id,
                    dngc,
                    "after_sequence",
                    existing_sequence + after_sequence,
                )
            )
    return pd.DataFrame(rows)


ROOT = repo_root_from_here()
DATA_DIR = ROOT / "data" / "skin_samples"
FRAGMENTS_CSV = (
    DATA_DIR
    / "top_fragments"
    / f"fragments_in_top_{TOP_CLUSTER_COUNT}_clusters_by_dngc.csv"
)
PARENT_SEQUENCES_CSV = DATA_DIR / "sequences.csv"
PROFILES_CSV = latest_profile_csv(DATA_DIR)
OUTPUT_CSV = DATA_DIR / "top_fragments" / "fragment_sequences_with_flanks.csv"

fragments = pd.read_csv(FRAGMENTS_CSV)
require_columns(
    fragments,
    {"id", "sequence", "name", "cluster", "position", "DNGC"},
    FRAGMENTS_CSV,
)
parent_frame = pd.read_csv(PARENT_SEQUENCES_CSV, usecols=["id", "sequence"])
parent_frame["id"] = parent_frame["id"].astype(str)
parent_sequences = parent_frame.drop_duplicates("id").set_index("id")["sequence"].astype(str)
parent_dngc = load_parent_dngc(PROFILES_CSV)

fragment_sequences = fragment_sequence_rows(fragments, parent_sequences, parent_dngc)
print(
    f"Loaded {len(fragments):,} fragments; created "
    f"{len(fragment_sequences):,} fragment/flank sequences."
)
display(fragment_sequences.head(30))

if SAVE_CSV:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    fragment_sequences.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote {OUTPUT_CSV}")

Loaded 205 fragments; created 588 fragment/flank sequences.


,parent_name,parent_id,cluster,DNGC,fragment_DNGC,fragment_id,position,sequence_type,sequence,length
0,Cathelicidin antimicrobial peptide,sp|P49913|CAMP_HUMAN,30,1.579821,1.5798,sp|P49913|CAMP_HUMAN-135,135–170,before_sequence,CDKDNKRFALLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES,46
1,Cathelicidin antimicrobial peptide,sp|P49913|CAMP_HUMAN,30,1.579821,1.5798,sp|P49913|CAMP_HUMAN-135,135–170,existing_sequence,LGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES,36
2,RNA-binding protein 8A,sp|Q9Y5S9|RBM8A_HUMAN,15,1.253120,1.2531,sp|Q9Y5S9|RBM8A_HUMAN-154,154–174,before_sequence,ISVDWCFVRGPPKGKRRGGRRRSRSPDRRRR,31
3,RNA-binding protein 8A,sp|Q9Y5S9|RBM8A_HUMAN,15,1.253120,1.2531,sp|Q9Y5S9|RBM8A_HUMAN-154,154–174,existing_sequence,PPKGKRRGGRRRSRSPDRRRR,21
4,LLGL scribble cell polarity complex component 2,sp|Q6P1M3|L2GL2_HUMAN,65,0.777287,1.1492,sp|Q6P1M3|L2GL2_HUMAN-633,633–668,before_sequence,FVKCTLHPSDQLALEGPLSRVKSLKKSLRQSFRRMRRSRVSSRKRH,46
5,LLGL scribble cell polarity complex component 2,sp|Q6P1M3|L2GL2_HUMAN,65,0.777287,1.1492,sp|Q6P1M3|L2GL2_HUMAN-633,633–668,existing_sequence,QLALEGPLSRVKSLKKSLRQSFRRMRRSRVSSRKRH,36
6,LLGL scribble cell polarity complex component 2,sp|Q6P1M3|L2GL2_HUMAN,65,0.777287,1.1492,sp|Q6P1M3|L2GL2_HUMAN-633,633–668,after_sequence,QLALEGPLSRVKSLKKSLRQSFRRMRRSRVSSRKRHPAGPPGEAQE,46
7,Probable ATP-dependent RNA helicase DDX47,sp|Q9H0S4|DDX47_HUMAN,22,1.140970,1.1410,sp|Q9H0S4|DDX47_HUMAN-206,206–237,before_sequence,PRDRKTFLFSATMTKKVQKLQRAALKNPVKCAVSSKYQTVEK,42
8,Probable ATP-dependent RNA helicase DDX47,sp|Q9H0S4|DDX47_HUMAN,22,1.140970,1.1410,sp|Q9H0S4|DDX47_HUMAN-206,206–237,existing_sequence,ATMTKKVQKLQRAALKNPVKCAVSSKYQTVEK,32
9,Probable ATP-dependent RNA helicase DDX47,sp|Q9H0S4|DDX47_HUMAN,22,1.140970,1.1410,sp|Q9H0S4|DDX47_HUMAN-206,206–237,after_sequence,ATMTKKVQKLQRAALKNPVKCAVSSKYQTVEKLQQYYIFIPS,42


Wrote /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/data/skin_samples/top_fragments/fragment_sequences_with_flanks.csv


Set `SAVE_CSV = False` to build and display the table without writing the output file. Each fragment contributes its existing sequence, a variant prepended with up to 10 preceding amino acids, and a variant appended with up to 10 following amino acids.